In [1]:
# ── CELL 1: Install RAPIDS + dependencies ──────────────────────────────────
import subprocess, sys
deps = [
    '--extra-index-url=https://pypi.nvidia.com',
    'cudf-cu12', 'cuml-cu12', 'dask-cuda', 'dask-cudf-cu12', 'xgboost',
    'fastapi', 'uvicorn[standard]', 'pyngrok'
    # NOTE: nest-asyncio intentionally NOT installed — it breaks LocalCUDACluster
]
cmd = [sys.executable, '-m', 'pip', 'install'] + deps
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\n✅ Done.' if proc.returncode == 0 else '\n❌ Failed.')

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com

✅ Done.


In [2]:
# ── CELL 2: Upload your CSV ─────────────────────────────────────────────────
from google.colab import files
print('Upload your 2025.csv file:')
uploaded = files.upload()
print('✅ Uploaded:', list(uploaded.keys()))

Upload your 2025.csv file:


Saving Agri008_1.xls to Agri008_1.xls
✅ Uploaded: ['Agri008_1.xls']


In [3]:
# ── CELL 3: Write app.py ────────────────────────────────────────────────────
import os; os.makedirs('templates', exist_ok=True)
app_code = r'''
import cudf
import cupy as cp
import dask_cudf
from dask_cuda import LocalCUDACluster
from dask.distributed import Client, wait
import dask
import xgboost as xgb
from xgboost import dask as dxgb
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse, JSONResponse
import os

FILE_PATH   = "2025.csv"
MONTH_NAMES = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
MONTH_DAYS  = {1:31,2:28,3:31,4:30,5:31,6:30,
               7:31,8:31,9:30,10:31,11:30,12:31}

app     = FastAPI(title="AgriCast")
client  = None
ddf_all = None


def init_cluster():
    # Boot cluster + load CSV. Called from launcher.py BEFORE uvicorn starts.
    global client, ddf_all
    print(">> Initialising LocalCUDACluster ...")
    cluster = LocalCUDACluster()
    client  = Client(cluster)
    print(f"   Dask ready -> {client.dashboard_link}")

    if not os.path.exists(FILE_PATH):
        raise RuntimeError(f"{FILE_PATH} not found.")

    print(f">> Loading {FILE_PATH} ...")
    ddf_all = dask_cudf.read_csv(FILE_PATH)
    ddf_all.columns = [c.strip() for c in ddf_all.columns]
    ddf_all["Arrival_Date"] = ddf_all["Arrival_Date"].map_partitions(
        cudf.to_datetime, format="%Y-%m-%d",
        meta=cudf.Series([], dtype="datetime64[ns]")
    )
    print(">> Data ready.")


def prepare_features(ddf):
    ddf["month"]        = ddf["Arrival_Date"].dt.month
    ddf["day"]          = ddf["Arrival_Date"].dt.day
    ddf["day_of_year"]  = ddf["Arrival_Date"].dt.dayofyear
    ddf["week_of_year"] = (ddf["day_of_year"] // 7) + 1

    def cyclic(df):
        df["sin_day"] = cp.sin(2 * cp.pi * df["day_of_year"].values / 365.0)
        df["cos_day"] = cp.cos(2 * cp.pi * df["day_of_year"].values / 365.0)
        return df

    ddf = ddf.map_partitions(cyclic)
    for lag in [1, 2, 3, 7]:
        ddf[f"lag_{lag}"] = ddf["Modal_Price"].shift(lag)
    return ddf.dropna()


FEATURES = ["month","day_of_year","week_of_year","sin_day","cos_day",
            "lag_1","lag_2","lag_3","lag_7"]


@app.get("/api/states")
async def get_states():
    states = sorted(ddf_all["State"].unique().compute().to_arrow().to_pylist())
    return JSONResponse({"states": states})


@app.get("/api/districts")
async def get_districts(state: str):
    mask = ddf_all["State"] == state
    districts = sorted(ddf_all[mask]["District"].unique().compute().to_arrow().to_pylist())
    return JSONResponse({"districts": districts})


@app.get("/api/commodities")
async def get_commodities(state: str, district: str):
    mask = (ddf_all["State"] == state) & (ddf_all["District"] == district)
    commodities = sorted(ddf_all[mask]["Commodity"].unique().compute().to_arrow().to_pylist())
    return JSONResponse({"commodities": commodities})


@app.get("/api/predict")
async def predict(state: str, district: str, commodity: str, month: int):
    global ddf_all, client
    if month < 1 or month > 12:
        raise HTTPException(400, "month must be 1-12")

    mask     = ((ddf_all["State"]     == state) &
                (ddf_all["District"]  == district) &
                (ddf_all["Commodity"] == commodity))
    hist_ddf = ddf_all[mask].repartition(npartitions=1)
    row_count = len(hist_ddf)
    if row_count < 10:
        raise HTTPException(400, f"Not enough data: only {row_count} rows.")

    ddf_proc = prepare_features(hist_ddf)
    X = ddf_proc[FEATURES].astype("float32")
    y = ddf_proc["Modal_Price"].astype("float32")
    X, y = dask.persist(X, y)
    wait([X, y])

    month_mask  = ddf_proc["month"] == month
    hist_month  = ddf_proc[month_mask][["day","Modal_Price","Min_Price","Max_Price"]].compute()
    days_2025   = hist_month["day"].to_arrow().to_pylist()
    actual_2025 = hist_month["Modal_Price"].to_arrow().to_pylist()

    all_prices = ddf_proc["Modal_Price"].compute()
    price_mean = float(all_prices.mean())
    price_std  = float(all_prices.std())
    price_min  = float(all_prices.min())
    price_max  = float(all_prices.max())

    dtrain = dxgb.DaskDMatrix(client, X, y)
    output = dxgb.train(
        client,
        {"tree_method":"hist","device":"cuda","objective":"reg:squarederror",
         "eta":0.05,"max_depth":6,"subsample":0.8},
        dtrain, num_boost_round=150
    )

    days       = list(range(1, MONTH_DAYS[month] + 1))
    date_strs  = [f"2026-{month:02d}-{d:02d}" for d in days]
    seed_price = float(hist_ddf["Modal_Price"].mean().compute())

    future_cdf = cudf.DataFrame({
        "Arrival_Date": cudf.to_datetime(cudf.Series(date_strs), format="%Y-%m-%d"),
        "Modal_Price" : cudf.Series([seed_price] * len(days), dtype="float32"),
        "Min_Price"   : cudf.Series([seed_price] * len(days), dtype="float32"),
        "Max_Price"   : cudf.Series([seed_price] * len(days), dtype="float32"),
    })
    future_ddf  = dask_cudf.from_cudf(future_cdf, npartitions=1)
    future_proc = prepare_features(future_ddf)
    X_future    = future_proc[FEATURES].astype("float32")
    preds_raw   = dxgb.predict(client, output, X_future).compute()
    days_2026   = future_proc["day"].compute().to_arrow().to_pylist()
    preds_2026  = [round(float(v), 2) for v in cp.asnumpy(preds_raw)]

    device     = cp.cuda.Device(0)
    gpu_name   = cp.cuda.runtime.getDeviceProperties(device.id)["name"].decode()
    rapids_ver = cudf.__version__

    return JSONResponse({
        "meta": {"state": state, "district": district, "commodity": commodity,
                 "month": MONTH_NAMES[month-1], "rows_trained": row_count,
                 "gpu": gpu_name, "rapids_version": rapids_ver},
        "stats": {"mean": round(price_mean,2), "std": round(price_std,2),
                  "min":  round(price_min,2),  "max": round(price_max,2)},
        "actual_2025":   {"days": days_2025,  "prices": [round(float(p),2) for p in actual_2025]},
        "forecast_2026": {"days": days_2026,  "prices": preds_2026}
    })


@app.get("/", response_class=HTMLResponse)
async def serve_ui():
    with open("templates/index.html") as f:
        return f.read()

'''
with open('app.py', 'w') as f:
    f.write(app_code)
print('✅ app.py written.')

✅ app.py written.


In [4]:
# ── CELL 4: Write templates/index.html ─────────────────────────────────────
html_code = r'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1.0"/>
<title>AgriCast — RAPIDS Price Forecaster</title>
<link rel="preconnect" href="https://fonts.googleapis.com"/>
<link href="https://fonts.googleapis.com/css2?family=Cormorant+Garamond:ital,wght@0,300;0,400;0,600;0,700;1,300;1,400&family=Syne:wght@400;500;600;700;800&family=IBM+Plex+Mono:wght@300;400;500&display=swap" rel="stylesheet"/>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
*{margin:0;padding:0;box-sizing:border-box}
:root{
  --bg-deep:#0D0F0A;
  --bg-mid:#141710;
  --bg-panel:#1A1D16;
  --bg-card:#1F2219;
  --bg-hover:#252920;
  --border-dim:rgba(255,255,255,0.06);
  --border-mid:rgba(255,255,255,0.10);
  --border-bright:rgba(255,255,255,0.16);
  --gold:#C9A84C;
  --gold-dim:#8A6E2F;
  --gold-pale:rgba(201,168,76,0.08);
  --sage:#7A9E7E;
  --sage-dim:#4A6B4E;
  --sage-pale:rgba(122,158,126,0.1);
  --rust:#B5541A;
  --text-primary:#F0EDE6;
  --text-secondary:#9C9882;
  --text-dim:#5A5845;
  --text-accent:#D4B96A;
  --mono:'IBM Plex Mono',monospace;
  --serif:'Cormorant Garamond',serif;
  --sans:'Syne',sans-serif;
}
html,body{height:100%;overflow:hidden}
body{font-family:var(--sans);background:var(--bg-deep);color:var(--text-primary);display:flex;flex-direction:column;height:100vh;position:relative;}

/* BOTANICAL BG */
.bg-botanical{position:fixed;inset:0;pointer-events:none;z-index:0;overflow:hidden;}
.bg-botanical svg{position:absolute;opacity:0.04;}
.svg-tl{top:-80px;left:-80px;width:560px;transform:rotate(-12deg)}
.svg-br{bottom:-100px;right:-100px;width:620px;transform:rotate(168deg)}
.svg-tr{top:15%;right:-50px;width:260px;transform:rotate(18deg)}
.grain{position:fixed;inset:0;pointer-events:none;z-index:1;background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='200' height='200'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.9' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='200' height='200' filter='url(%23n)' opacity='0.04'/%3E%3C/svg%3E");opacity:0.5;}

/* HEADER */
header{position:relative;z-index:10;background:rgba(13,15,10,0.94);backdrop-filter:blur(16px);border-bottom:1px solid var(--border-dim);padding:0 2rem;display:flex;align-items:center;justify-content:space-between;height:60px;flex-shrink:0;}
.logo-mark{display:flex;align-items:center;gap:14px;}
.logo-glyph{width:30px;height:30px;flex-shrink:0;}
.logo-name{font-family:var(--serif);font-size:1.3rem;font-weight:600;color:var(--text-primary);letter-spacing:0.04em;line-height:1;}
.logo-tagline{font-family:var(--mono);font-size:8.5px;color:var(--text-dim);letter-spacing:0.22em;text-transform:uppercase;margin-top:2px;}
.header-chips{display:flex;gap:6px;}
.hchip{font-family:var(--mono);font-size:9px;letter-spacing:0.14em;padding:4px 11px;border-radius:2px;text-transform:uppercase;display:flex;align-items:center;gap:5px;}
.hchip-a{background:rgba(122,158,126,0.1);color:var(--sage);border:1px solid rgba(122,158,126,0.2);}
.hchip-b{background:var(--gold-pale);color:var(--gold);border:1px solid rgba(201,168,76,0.2);}
.pulse{width:5px;height:5px;border-radius:50%;background:currentColor;animation:blink 2.4s ease-in-out infinite;}
@keyframes blink{0%,100%{opacity:1}50%{opacity:0.2}}

/* LAYOUT */
.layout{display:grid;grid-template-columns:300px 1fr;flex:1;overflow:hidden;position:relative;z-index:2;}

/* SIDEBAR */
.sidebar{background:var(--bg-panel);border-right:1px solid var(--border-dim);display:flex;flex-direction:column;overflow-y:auto;overflow-x:hidden;}
.sidebar::-webkit-scrollbar{width:3px}
.sidebar::-webkit-scrollbar-thumb{background:var(--border-mid)}
.s-pad{padding:1.5rem}
.s-pad-x{padding:0 1.5rem}
.eyebrow{font-family:var(--mono);font-size:9px;letter-spacing:0.22em;text-transform:uppercase;color:var(--text-dim);margin-bottom:1rem;display:flex;align-items:center;gap:8px;}
.eyebrow::after{content:'';flex:1;height:1px;background:var(--border-dim);}
.fields{display:flex;flex-direction:column;gap:1rem;padding:0 1.5rem;}
.field{display:flex;flex-direction:column;gap:5px;}
.field-label{font-family:var(--mono);font-size:9px;letter-spacing:0.18em;text-transform:uppercase;color:var(--text-secondary);}
select{width:100%;background:var(--bg-card);border:1px solid var(--border-mid);border-radius:3px;color:var(--text-primary);font-family:var(--sans);font-size:13px;padding:9px 32px 9px 12px;appearance:none;cursor:pointer;transition:border-color .2s,background .2s;background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='10' height='6'%3E%3Cpath d='M0 0l5 6 5-6z' fill='%235A5845'/%3E%3C/svg%3E");background-repeat:no-repeat;background-position:right 12px center;}
select:focus{outline:none;border-color:var(--gold-dim);background:var(--bg-hover);}
select:disabled{opacity:0.3;cursor:not-allowed;}
select option{background:var(--bg-card);}
.divider-line{height:1px;background:var(--border-dim);margin:1.25rem 1.5rem;}
.model-info{padding:0 1.5rem;}
.model-rows{display:flex;flex-direction:column;gap:0;}
.model-row{display:flex;justify-content:space-between;align-items:baseline;padding:5px 0;border-bottom:1px solid var(--border-dim);}
.model-row:last-child{border-bottom:none;}
.model-k{font-family:var(--mono);font-size:9.5px;color:var(--text-dim);}
.model-v{font-family:var(--mono);font-size:9.5px;color:var(--sage);}
.sidebar-bottom{margin-top:auto;padding:1.5rem;}
.btn-run{width:100%;background:transparent;border:1px solid var(--gold-dim);border-radius:3px;color:var(--gold);font-family:var(--sans);font-size:13px;font-weight:600;letter-spacing:0.1em;text-transform:uppercase;padding:13px;cursor:pointer;transition:background .2s,border-color .2s,color .2s;}
.btn-run:hover:not(:disabled){border-color:var(--gold);color:var(--text-primary);background:var(--gold-pale);}
.btn-run:active:not(:disabled){transform:scale(0.99);}
.btn-run:disabled{opacity:0.3;cursor:not-allowed;}
.gpu-panel{background:rgba(122,158,126,0.04);border:1px solid rgba(122,158,126,0.1);border-radius:3px;padding:1rem;margin-top:0.75rem;display:none;}
.gpu-panel-title{font-family:var(--mono);font-size:9px;letter-spacing:0.2em;color:var(--sage-dim);text-transform:uppercase;margin-bottom:8px;}
.gpu-row-item{display:flex;justify-content:space-between;margin-bottom:4px;}
.gpu-k{font-family:var(--mono);font-size:9.5px;color:var(--text-dim);}
.gpu-v{font-family:var(--mono);font-size:9.5px;color:var(--sage);font-weight:500;}

/* CONTENT */
.content{overflow-y:auto;overflow-x:hidden;background:var(--bg-deep);}
.content::-webkit-scrollbar{width:3px}
.content::-webkit-scrollbar-thumb{background:var(--border-mid)}

/* HERO */
.hero{min-height:100%;display:flex;flex-direction:column;align-items:center;justify-content:center;padding:4rem 3rem;text-align:center;gap:1.5rem;}
.hero-rule{width:1px;height:64px;background:linear-gradient(to bottom,transparent,var(--gold-dim),transparent);margin:0 auto;}
.hero-headline{font-family:var(--serif);font-size:3.2rem;font-weight:300;font-style:italic;color:var(--text-primary);line-height:1.15;max-width:460px;}
.hero-headline strong{font-style:normal;font-weight:700;display:block;color:var(--text-accent);}
.hero-body{font-family:var(--mono);font-size:10.5px;color:var(--text-dim);letter-spacing:0.15em;line-height:2;text-transform:uppercase;}
.hero-steps{display:flex;border:1px solid var(--border-dim);border-radius:3px;overflow:hidden;margin-top:0.5rem;}
.hero-step{padding:9px 18px;font-family:var(--mono);font-size:9px;color:var(--text-dim);letter-spacing:0.12em;text-transform:uppercase;border-right:1px solid var(--border-dim);}
.hero-step:last-child{border-right:none;}

/* LOADING */
.loading{min-height:100%;display:none;flex-direction:column;align-items:center;justify-content:center;gap:2rem;}
.loading-ring{width:56px;height:56px;border:1px solid var(--border-mid);border-top-color:var(--gold);border-radius:50%;animation:spin 1.4s linear infinite;}
@keyframes spin{to{transform:rotate(360deg)}}
.loading-label{font-family:var(--mono);font-size:10px;letter-spacing:0.24em;color:var(--text-dim);text-transform:uppercase;animation:fadeCycle 1.8s ease-in-out infinite;}
@keyframes fadeCycle{0%,100%{opacity:0.4}50%{opacity:1}}

/* RESULTS */
.results{display:none;flex-direction:column;}

/* BAND */
.result-band{background:var(--bg-panel);border-bottom:1px solid var(--border-dim);padding:1.1rem 2rem;display:flex;align-items:center;flex-wrap:wrap;gap:0;}
.band-item{display:flex;flex-direction:column;gap:2px;padding:0 1.5rem;border-right:1px solid var(--border-dim);}
.band-item:first-child{padding-left:0;}
.band-item:last-child{border-right:none;}
.band-k{font-family:var(--mono);font-size:8px;letter-spacing:0.2em;color:var(--text-dim);text-transform:uppercase;}
.band-v{font-family:var(--serif);font-size:1.05rem;font-weight:600;color:var(--text-primary);}
.band-gpu{margin-left:auto;display:flex;gap:6px;padding-left:1.5rem;}
.band-chip{font-family:var(--mono);font-size:9px;letter-spacing:0.1em;padding:4px 10px;border-radius:2px;text-transform:uppercase;}
.bc-sage{background:rgba(122,158,126,0.1);color:var(--sage);border:1px solid rgba(122,158,126,0.2);}
.bc-gold{background:var(--gold-pale);color:var(--gold);border:1px solid rgba(201,168,76,0.2);}

.result-body{padding:1.75rem 2rem;display:flex;flex-direction:column;gap:1.75rem;}

/* STATS */
.stats-row{display:grid;grid-template-columns:repeat(4,1fr);gap:1px;background:var(--border-dim);border:1px solid var(--border-dim);border-radius:4px;overflow:hidden;}
.stat-card{background:var(--bg-card);padding:1.25rem 1.4rem;position:relative;transition:background .2s;}
.stat-card:hover{background:var(--bg-hover);}
.stat-accent{position:absolute;top:0;left:0;right:0;height:2px;}
.stat-k{font-family:var(--mono);font-size:9px;letter-spacing:0.18em;text-transform:uppercase;color:var(--text-dim);margin-bottom:10px;}
.stat-v{font-family:var(--serif);font-size:2rem;font-weight:300;color:var(--text-primary);line-height:1;}
.stat-sub{font-family:var(--mono);font-size:9px;color:var(--text-dim);margin-top:5px;}

/* CHARTS */
.charts-grid{display:grid;grid-template-columns:1fr 1fr;gap:1px;background:var(--border-dim);border:1px solid var(--border-dim);border-radius:4px;overflow:hidden;}
.chart-card{background:var(--bg-card);padding:1.5rem;}
.chart-card.wide{grid-column:1/-1;border-bottom:1px solid var(--border-dim);}
.chart-title{font-family:var(--serif);font-size:1.05rem;font-style:italic;font-weight:400;color:var(--text-primary);}
.chart-sub{font-family:var(--mono);font-size:9px;letter-spacing:0.14em;color:var(--text-dim);text-transform:uppercase;margin-top:3px;margin-bottom:1.2rem;}
.chart-wrap{position:relative;height:260px;}
.chart-wrap-sm{position:relative;height:210px;}

/* TABLE */
.table-section{border:1px solid var(--border-dim);border-radius:4px;overflow:hidden;background:var(--bg-card);}
.table-head{padding:1rem 1.5rem;border-bottom:1px solid var(--border-dim);background:var(--bg-panel);display:flex;align-items:baseline;justify-content:space-between;}
.table-head-title{font-family:var(--serif);font-size:1rem;font-style:italic;color:var(--text-primary);}
.table-head-sub{font-family:var(--mono);font-size:9px;letter-spacing:0.14em;color:var(--text-dim);text-transform:uppercase;}
.table-scroll{overflow-x:auto;max-height:300px;overflow-y:auto;}
.table-scroll::-webkit-scrollbar{width:3px;height:3px}
.table-scroll::-webkit-scrollbar-thumb{background:var(--border-mid)}
table{width:100%;border-collapse:collapse;}
thead{position:sticky;top:0;z-index:2;background:var(--bg-panel);}
th{padding:8px 14px;text-align:left;font-family:var(--mono);font-size:9px;letter-spacing:0.18em;color:var(--text-dim);text-transform:uppercase;border-bottom:1px solid var(--border-dim);font-weight:400;}
td{padding:9px 14px;font-size:12.5px;border-bottom:1px solid rgba(255,255,255,0.03);color:var(--text-secondary);transition:background .15s;}
tr:hover td{background:rgba(255,255,255,0.02);color:var(--text-primary);}
tr:last-child td{border-bottom:none;}
.td-day{font-family:var(--mono);font-size:10.5px;color:var(--text-dim);}
.td-actual{font-family:var(--serif);font-size:1rem;color:var(--sage);}
.td-fore{font-family:var(--serif);font-size:1rem;font-weight:600;color:var(--text-primary);}
.td-up{font-family:var(--mono);font-size:10px;color:var(--rust);}
.td-dn{font-family:var(--mono);font-size:10px;color:var(--sage);}
.td-eq{font-family:var(--mono);font-size:10px;color:var(--text-dim);}
.dir-up{font-family:var(--mono);font-size:9.5px;color:var(--rust);letter-spacing:0.08em;}
.dir-dn{font-family:var(--mono);font-size:9.5px;color:var(--sage);letter-spacing:0.08em;}
.dir-eq{font-family:var(--mono);font-size:9.5px;color:var(--text-dim);letter-spacing:0.08em;}
</style>
</head>
<body>

<!-- BOTANICAL BACKGROUND -->
<div class="bg-botanical">
  <svg class="svg-tl" viewBox="0 0 500 500" fill="none" xmlns="http://www.w3.org/2000/svg">
    <line x1="250" y1="500" x2="250" y2="50" stroke="white" stroke-width="1.2"/>
    <line x1="200" y1="500" x2="200" y2="100" stroke="white" stroke-width="1"/>
    <line x1="300" y1="500" x2="300" y2="80" stroke="white" stroke-width="1"/>
    <line x1="150" y1="500" x2="150" y2="160" stroke="white" stroke-width="0.7"/>
    <line x1="350" y1="500" x2="350" y2="140" stroke="white" stroke-width="0.7"/>
    <ellipse cx="250" cy="58" rx="9" ry="20" stroke="white" stroke-width="1"/>
    <ellipse cx="250" cy="40" rx="6" ry="13" stroke="white" stroke-width="0.8"/>
    <ellipse cx="200" cy="108" rx="8" ry="17" stroke="white" stroke-width="1"/>
    <ellipse cx="300" cy="88" rx="8" ry="17" stroke="white" stroke-width="1"/>
    <ellipse cx="150" cy="168" rx="6" ry="14" stroke="white" stroke-width="0.7"/>
    <ellipse cx="350" cy="148" rx="6" ry="14" stroke="white" stroke-width="0.7"/>
    <path d="M250 195 Q272 180 288 198" stroke="white" stroke-width="0.8" fill="none"/>
    <path d="M250 195 Q228 180 212 198" stroke="white" stroke-width="0.8" fill="none"/>
    <path d="M250 275 Q276 258 294 278" stroke="white" stroke-width="0.7" fill="none"/>
    <path d="M200 235 Q222 218 236 238" stroke="white" stroke-width="0.7" fill="none"/>
    <path d="M300 255 Q320 238 332 258" stroke="white" stroke-width="0.7" fill="none"/>
    <circle cx="75" cy="340" r="45" stroke="white" stroke-width="0.5" fill="none"/>
    <circle cx="75" cy="340" r="30" stroke="white" stroke-width="0.35" fill="none"/>
    <circle cx="425" cy="175" r="32" stroke="white" stroke-width="0.5" fill="none"/>
    <line x1="0" y1="420" x2="500" y2="420" stroke="white" stroke-width="0.3" opacity="0.5"/>
    <line x1="0" y1="460" x2="500" y2="460" stroke="white" stroke-width="0.2" opacity="0.3"/>
    <line x1="100" y1="0" x2="100" y2="500" stroke="white" stroke-width="0.3" opacity="0.3"/>
    <line x1="400" y1="0" x2="400" y2="500" stroke="white" stroke-width="0.2" opacity="0.25"/>
  </svg>
  <svg class="svg-br" viewBox="0 0 500 500" fill="none" xmlns="http://www.w3.org/2000/svg">
    <line x1="250" y1="500" x2="250" y2="50" stroke="white" stroke-width="1.2"/>
    <line x1="190" y1="500" x2="190" y2="110" stroke="white" stroke-width="1"/>
    <line x1="310" y1="500" x2="310" y2="85" stroke="white" stroke-width="1"/>
    <ellipse cx="250" cy="58" rx="9" ry="20" stroke="white" stroke-width="1"/>
    <ellipse cx="190" cy="118" rx="8" ry="17" stroke="white" stroke-width="1"/>
    <ellipse cx="310" cy="93" rx="8" ry="17" stroke="white" stroke-width="1"/>
    <path d="M250 195 Q272 180 288 198" stroke="white" stroke-width="0.8" fill="none"/>
    <path d="M250 195 Q228 180 212 198" stroke="white" stroke-width="0.8" fill="none"/>
    <circle cx="95" cy="390" r="60" stroke="white" stroke-width="0.5" fill="none"/>
    <circle cx="95" cy="390" r="42" stroke="white" stroke-width="0.35" fill="none"/>
    <circle cx="390" cy="110" r="42" stroke="white" stroke-width="0.4" fill="none"/>
    <line x1="0" y1="90" x2="500" y2="90" stroke="white" stroke-width="0.3" opacity="0.4"/>
    <line x1="155" y1="0" x2="155" y2="500" stroke="white" stroke-width="0.2" opacity="0.3"/>
  </svg>
  <svg class="svg-tr" viewBox="0 0 260 380" fill="none" xmlns="http://www.w3.org/2000/svg">
    <circle cx="130" cy="190" r="88" stroke="white" stroke-width="0.5" fill="none"/>
    <circle cx="130" cy="190" r="62" stroke="white" stroke-width="0.4" fill="none"/>
    <circle cx="130" cy="190" r="38" stroke="white" stroke-width="0.3" fill="none"/>
    <line x1="130" y1="102" x2="130" y2="278" stroke="white" stroke-width="0.4"/>
    <line x1="42" y1="190" x2="218" y2="190" stroke="white" stroke-width="0.4"/>
    <line x1="68" y1="128" x2="192" y2="252" stroke="white" stroke-width="0.3"/>
    <line x1="192" y1="128" x2="68" y2="252" stroke="white" stroke-width="0.3"/>
  </svg>
</div>
<div class="grain"></div>

<!-- HEADER -->
<header>
  <div class="logo-mark">
    <svg class="logo-glyph" viewBox="0 0 30 30" fill="none" xmlns="http://www.w3.org/2000/svg">
      <line x1="15" y1="28" x2="15" y2="7" stroke="#C9A84C" stroke-width="1.2"/>
      <line x1="11" y1="28" x2="11" y2="11" stroke="#C9A84C" stroke-width="0.9" opacity="0.55"/>
      <line x1="19" y1="28" x2="19" y2="9" stroke="#C9A84C" stroke-width="0.9" opacity="0.55"/>
      <ellipse cx="15" cy="6.5" rx="3.2" ry="6.5" stroke="#C9A84C" stroke-width="1"/>
      <ellipse cx="11" cy="10" rx="2.8" ry="5.5" stroke="#C9A84C" stroke-width="0.8" opacity="0.55"/>
      <ellipse cx="19" cy="8" rx="2.8" ry="5.5" stroke="#C9A84C" stroke-width="0.8" opacity="0.55"/>
      <path d="M15 15 Q18.5 12.5 21 15" stroke="#C9A84C" stroke-width="0.7" fill="none" opacity="0.5"/>
      <path d="M15 15 Q11.5 12.5 9 15" stroke="#C9A84C" stroke-width="0.7" fill="none" opacity="0.5"/>
    </svg>
    <div>
      <div class="logo-name">AgriCast</div>
      <div class="logo-tagline">NVIDIA RAPIDS · GPU Price Intelligence</div>
    </div>
  </div>
  <div class="header-chips">
    <div class="hchip hchip-a"><span class="pulse"></span>CUDA Active</div>
    <div class="hchip hchip-b">RAPIDS</div>
  </div>
</header>

<!-- LAYOUT -->
<div class="layout">
  <!-- SIDEBAR -->
  <aside class="sidebar">
    <div class="s-pad">
      <div class="eyebrow">Data Filters</div>
    </div>
    <div class="fields">
      <div class="field">
        <div class="field-label">State</div>
        <select id="sel-state" onchange="loadDistricts()"><option value="">Loading…</option></select>
      </div>
      <div class="field">
        <div class="field-label">District</div>
        <select id="sel-district" disabled onchange="loadCommodities()"><option value="">Select state first</option></select>
      </div>
      <div class="field">
        <div class="field-label">Commodity</div>
        <select id="sel-commodity" disabled><option value="">Select district first</option></select>
      </div>
      <div class="field">
        <div class="field-label">Forecast Month — 2026</div>
        <select id="sel-month">
          <option value="1">January</option><option value="2">February</option>
          <option value="3">March</option><option value="4">April</option>
          <option value="5">May</option><option value="6">June</option>
          <option value="7">July</option><option value="8">August</option>
          <option value="9">September</option><option value="10">October</option>
          <option value="11">November</option><option value="12">December</option>
        </select>
      </div>
    </div>

    <div class="divider-line"></div>

    <div class="model-info">
      <div class="eyebrow">Model Config</div>
      <div class="model-rows">
        <div class="model-row"><span class="model-k">Engine</span><span class="model-v">XGBoost CUDA</span></div>
        <div class="model-row"><span class="model-k">Tree Method</span><span class="model-v">hist</span></div>
        <div class="model-row"><span class="model-k">Boost Rounds</span><span class="model-v">150</span></div>
        <div class="model-row"><span class="model-k">Encoding</span><span class="model-v">sin / cos cyclic</span></div>
        <div class="model-row"><span class="model-k">Lag Features</span><span class="model-v">1, 2, 3, 7 days</span></div>
        <div class="model-row"><span class="model-k">Data Layer</span><span class="model-v">cuDF · Dask-cuDF</span></div>
      </div>
    </div>

    <div class="sidebar-bottom">
      <button class="btn-run" id="btn-run" onclick="runForecast()" disabled>Generate Forecast</button>
      <div class="gpu-panel" id="gpu-panel">
        <div class="gpu-panel-title">Compute Info</div>
        <div class="gpu-row-item"><span class="gpu-k">GPU</span><span class="gpu-v" id="gpu-name">—</span></div>
        <div class="gpu-row-item"><span class="gpu-k">RAPIDS</span><span class="gpu-v" id="rapids-ver">—</span></div>
        <div class="gpu-row-item"><span class="gpu-k">Rows Trained</span><span class="gpu-v" id="rows-trained">—</span></div>
      </div>
    </div>
  </aside>

  <!-- CONTENT -->
  <main class="content" id="content">

    <!-- HERO -->
    <div class="hero" id="hero">
      <div class="hero-rule"></div>
      <div class="hero-headline">
        <strong>GPU-Accelerated</strong>
        Agricultural Price Forecasting
      </div>
      <div class="hero-body">
        cuDF · Dask-cuDF · XGBoost CUDA<br/>
        All computation in GPU memory<br/>
        Zero NumPy · Zero Pandas · Pure RAPIDS
      </div>
      <div class="hero-steps">
        <div class="hero-step">01 · State</div>
        <div class="hero-step">02 · District</div>
        <div class="hero-step">03 · Commodity</div>
        <div class="hero-step">04 · Forecast</div>
      </div>
    </div>

    <!-- LOADING -->
    <div class="loading" id="loading">
      <div class="loading-ring"></div>
      <div class="loading-label" id="loading-msg">Initialising GPU Pipeline</div>
    </div>

    <!-- RESULTS -->
    <div class="results" id="results">
      <div class="result-band" id="result-band"></div>
      <div class="result-body">
        <div class="stats-row" id="stats-row"></div>
        <div class="charts-grid">
          <div class="chart-card wide">
            <div class="chart-title">2025 Actuals vs 2026 Forecast — Modal Price</div>
            <div class="chart-sub">Daily prices · Rupees per quintal · GPU-predicted via XGBoost</div>
            <div class="chart-wrap"><canvas id="chart-line"></canvas></div>
          </div>
          <div class="chart-card">
            <div class="chart-title">Day-by-Day Forecast</div>
            <div class="chart-sub">2026 predicted prices</div>
            <div class="chart-wrap-sm"><canvas id="chart-bar"></canvas></div>
          </div>
          <div class="chart-card">
            <div class="chart-title">Historical Price Distribution</div>
            <div class="chart-sub">2025 frequency histogram</div>
            <div class="chart-wrap-sm"><canvas id="chart-hist"></canvas></div>
          </div>
        </div>
        <div class="table-section">
          <div class="table-head">
            <div class="table-head-title">Day-by-Day Comparison</div>
            <div class="table-head-sub">2025 Actual vs 2026 Forecast · INR / quintal</div>
          </div>
          <div class="table-scroll">
            <table>
              <thead><tr><th>Day</th><th>2025 Actual</th><th>2026 Forecast</th><th>Delta</th><th>Direction</th></tr></thead>
              <tbody id="table-body"></tbody>
            </table>
          </div>
        </div>
      </div>
    </div>

  </main>
</div>

<script>
let cLine=null,cBar=null,cHist=null;
Chart.defaults.font.family="'IBM Plex Mono',monospace";
Chart.defaults.color='#5A5845';

const C={gold:'#C9A84C',goldA:'rgba(201,168,76,0.12)',goldAA:'rgba(201,168,76,0.06)',sage:'#7A9E7E',sageA:'rgba(122,158,126,0.12)',rust:'#B5541A',grid:'rgba(255,255,255,0.04)',tt:'#1F2219'};

const baseScale={grid:{color:C.grid},border:{color:'rgba(255,255,255,0.05)'},ticks:{font:{size:9},color:'#5A5845'}};
const baseOpts={
  responsive:true,maintainAspectRatio:false,
  plugins:{
    legend:{position:'top',labels:{usePointStyle:true,pointStyleWidth:7,padding:16,font:{size:9,family:"'IBM Plex Mono',monospace"},color:'#9C9882'}},
    tooltip:{backgroundColor:C.tt,borderColor:'rgba(255,255,255,0.07)',borderWidth:1,titleFont:{family:"'IBM Plex Mono',monospace",size:9},bodyFont:{family:"'Cormorant Garamond',serif",size:15},titleColor:'#5A5845',bodyColor:'#F0EDE6',padding:12,cornerRadius:3,callbacks:{label:ctx=>' \u20B9'+ctx.parsed.y.toLocaleString()}}
  },
  scales:{x:{...baseScale},y:{...baseScale,ticks:{...baseScale.ticks,callback:v=>'\u20B9'+v.toLocaleString()}}}
};

function deepClone(o){return JSON.parse(JSON.stringify(o))}
function destroyCharts(){[cLine,cBar,cHist].forEach(c=>{if(c)c.destroy()})}

function buildCharts(data){
  destroyCharts();
  const a=data.actual_2025,f=data.forecast_2026;

  cLine=new Chart(document.getElementById('chart-line'),{
    type:'line',
    data:{
      labels:f.days.map(d=>'D'+String(d).padStart(2,'0')),
      datasets:[
        {label:'2025 Actual',data:a.prices,borderColor:C.sage,backgroundColor:C.sageA,fill:true,tension:0.38,pointRadius:2,pointBackgroundColor:C.sage,borderWidth:1.8},
        {label:'2026 Forecast',data:f.prices,borderColor:C.gold,backgroundColor:C.goldAA,fill:true,tension:0.38,pointRadius:2,pointBackgroundColor:C.gold,borderWidth:2,borderDash:[5,4]}
      ]
    },options:deepClone(baseOpts)
  });

  const bc=f.prices.map((p,i)=>p>(f.prices[i-1]||p)?'rgba(181,84,26,0.65)':'rgba(122,158,126,0.65)');
  const bo=deepClone(baseOpts);bo.plugins.legend.display=false;
  cBar=new Chart(document.getElementById('chart-bar'),{
    type:'bar',
    data:{labels:f.days.map(d=>''+d),datasets:[{data:f.prices,backgroundColor:bc,borderColor:bc.map(c=>c.replace('0.65','1')),borderWidth:1,borderRadius:2,borderSkipped:false}]},
    options:bo
  });

  const pp=a.prices.filter(Boolean);
  if(pp.length>1){
    const mn=Math.min(...pp),mx=Math.max(...pp),bins=12,bw=(mx-mn)/bins;
    const bkts=Array(bins).fill(0),lbls=[];
    for(let i=0;i<bins;i++)lbls.push('\u20B9'+(mn+i*bw).toFixed(0));
    pp.forEach(p=>{let b=Math.floor((p-mn)/bw);if(b===bins)b=bins-1;bkts[b]++});
    const ho=deepClone(baseOpts);ho.plugins.legend.display=false;
    ho.scales.y.ticks.callback=v=>v;
    cHist=new Chart(document.getElementById('chart-hist'),{
      type:'bar',
      data:{labels:lbls,datasets:[{data:bkts,backgroundColor:'rgba(201,168,76,0.3)',borderColor:C.gold,borderWidth:1,borderRadius:2}]},
      options:ho
    });
  }
}

function buildTable(data){
  const a=data.actual_2025,f=data.forecast_2026;
  const tb=document.getElementById('table-body');tb.innerHTML='';
  const len=Math.max(a.days.length,f.days.length);
  for(let i=0;i<len;i++){
    const day=f.days[i]||a.days[i];
    const act=a.prices[i]!=null?a.prices[i]:null;
    const fore=f.prices[i]!=null?f.prices[i]:null;
    const diff=act!=null&&fore!=null?+(fore-act).toFixed(2):null;
    const pct=act&&diff!=null?((diff/act)*100).toFixed(1):null;
    const up=diff>0.5,dn=diff<-0.5;
    tb.innerHTML+=`<tr>
      <td class="td-day">${String(day).padStart(2,'0')}</td>
      <td class="td-actual">${act!=null?'\u20B9'+act.toLocaleString():'—'}</td>
      <td class="td-fore">\u20B9${fore!=null?fore.toLocaleString():'—'}</td>
      <td class="${up?'td-up':dn?'td-dn':'td-eq'}">${diff!=null?(diff>0?'+':'')+diff+(pct?' ('+pct+'%)':''):'—'}</td>
      <td class="${up?'dir-up':dn?'dir-dn':'dir-eq'}">${up?'Rising':dn?'Falling':'Stable'}</td>
    </tr>`;
  }
}

function buildStats(data){
  const s=data.stats,f=data.forecast_2026;
  const pm=(f.prices.reduce((a,b)=>a+b,0)/f.prices.length).toFixed(0);
  const pct=(((pm-s.mean)/s.mean)*100).toFixed(1);
  const cards=[
    {k:'2025 Mean Price',v:'\u20B9'+s.mean.toLocaleString(),sub:'historical average',acc:C.sage},
    {k:'2026 Forecast Avg',v:'\u20B9'+Number(pm).toLocaleString(),sub:(pct>0?'+':'')+pct+'% vs 2025',acc:C.gold},
    {k:'Price Range',v:'\u20B9'+(s.max-s.min).toLocaleString(),sub:'max \u2212 min spread',acc:'rgba(181,84,26,0.85)'},
    {k:'Std Deviation',v:'\u20B9'+s.std.toLocaleString(),sub:'price volatility',acc:'rgba(45,125,154,0.85)'},
  ];
  document.getElementById('stats-row').innerHTML=cards.map(c=>`
    <div class="stat-card">
      <div class="stat-accent" style="background:${c.acc}"></div>
      <div class="stat-k">${c.k}</div>
      <div class="stat-v">${c.v}</div>
      <div class="stat-sub">${c.sub}</div>
    </div>`).join('');
}

function buildBand(m){
  document.getElementById('result-band').innerHTML=`
    <div class="band-item"><div class="band-k">State</div><div class="band-v">${m.state}</div></div>
    <div class="band-item"><div class="band-k">District</div><div class="band-v">${m.district}</div></div>
    <div class="band-item"><div class="band-k">Commodity</div><div class="band-v">${m.commodity}</div></div>
    <div class="band-item"><div class="band-k">Month</div><div class="band-v">${m.month} 2026</div></div>
    <div class="band-item"><div class="band-k">Trained On</div><div class="band-v">${m.rows_trained.toLocaleString()} rows</div></div>
    <div class="band-gpu">
      <div class="band-chip bc-sage">${m.gpu}</div>
      <div class="band-chip bc-gold">RAPIDS v${m.rapids_version}</div>
    </div>`;
  const gp=document.getElementById('gpu-panel');gp.style.display='block';
  document.getElementById('gpu-name').textContent=m.gpu;
  document.getElementById('rapids-ver').textContent='v'+m.rapids_version;
  document.getElementById('rows-trained').textContent=m.rows_trained.toLocaleString();
}

const MSGS=['Loading CSV into GPU memory','Building Dask-cuDF partitions','Engineering lag & cyclic features','Initialising DaskDMatrix on CUDA','Training XGBoost — 150 rounds','Running inference on GPU','Assembling forecast response'];
let mt=null;
function startLoad(){let i=0;document.getElementById('loading-msg').textContent=MSGS[0];mt=setInterval(()=>{i=(i+1)%MSGS.length;document.getElementById('loading-msg').textContent=MSGS[i]},1900);}
function stopLoad(){clearInterval(mt)}

function show(id){document.getElementById(id).style.display='flex'}
function hide(id){document.getElementById(id).style.display='none'}
function showR(){document.getElementById('results').style.display='flex'}

async function loadStates(){
  const r=await fetch('/api/states');const d=await r.json();
  const s=document.getElementById('sel-state');
  s.innerHTML='<option value="">Select state</option>'+d.states.map(x=>`<option value="${x}">${x}</option>`).join('');
}
async function loadDistricts(){
  const st=document.getElementById('sel-state').value;if(!st)return;
  const ds=document.getElementById('sel-district'),cs=document.getElementById('sel-commodity');
  ds.innerHTML='<option>Loading…</option>';ds.disabled=true;
  cs.innerHTML='<option>Select district first</option>';cs.disabled=true;
  document.getElementById('btn-run').disabled=true;
  const r=await fetch(`/api/districts?state=${encodeURIComponent(st)}`);const d=await r.json();
  ds.innerHTML='<option value="">Select district</option>'+d.districts.map(x=>`<option value="${x}">${x}</option>`).join('');
  ds.disabled=false;
}
async function loadCommodities(){
  const st=document.getElementById('sel-state').value,di=document.getElementById('sel-district').value;
  if(!di)return;
  const cs=document.getElementById('sel-commodity');
  cs.innerHTML='<option>Loading…</option>';cs.disabled=true;
  document.getElementById('btn-run').disabled=true;
  const r=await fetch(`/api/commodities?state=${encodeURIComponent(st)}&district=${encodeURIComponent(di)}`);const d=await r.json();
  cs.innerHTML='<option value="">Select commodity</option>'+d.commodities.map(x=>`<option value="${x}">${x}</option>`).join('');
  cs.disabled=false;document.getElementById('btn-run').disabled=false;
}
async function runForecast(){
  const st=document.getElementById('sel-state').value,di=document.getElementById('sel-district').value,co=document.getElementById('sel-commodity').value,mo=document.getElementById('sel-month').value;
  if(!st||!di||!co){alert('Please complete all filters.');return}
  hide('hero');hide('results');show('loading');startLoad();
  document.getElementById('btn-run').disabled=true;
  try{
    const r=await fetch(`/api/predict?state=${encodeURIComponent(st)}&district=${encodeURIComponent(di)}&commodity=${encodeURIComponent(co)}&month=${mo}`);
    if(!r.ok){const e=await r.json();throw new Error(e.detail||'Prediction failed')}
    const data=await r.json();
    stopLoad();hide('loading');
    buildBand(data.meta);buildStats(data);buildCharts(data);buildTable(data);showR();
    document.getElementById('content').scrollTo({top:0,behavior:'smooth'});
  }catch(err){stopLoad();hide('loading');show('hero');alert('Error: '+err.message);}
  finally{document.getElementById('btn-run').disabled=false;}
}
loadStates();
</script>
</body>
</html>

'''
with open('templates/index.html', 'w') as f:
    f.write(html_code)
print('✅ index.html written.')

✅ index.html written.


In [5]:
# ── CELL 5: Write launcher.py ──────────────────────────────────────────────
launcher_code = r'''
#!/usr/bin/env python3
\"\"\"
launcher.py
Boots LocalCUDACluster synchronously in a clean process (no Jupyter loop),
then hands control to uvicorn which creates its own fresh event loop.
\"\"\"
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import app as app_module
app_module.init_cluster()          # cluster + CSV load — no event loop yet

import uvicorn
uvicorn.run(app_module.app, host="0.0.0.0", port=8000, log_level="info")

'''
with open('launcher.py', 'w') as f:
    f.write(launcher_code)
print('✅ launcher.py written.')

✅ launcher.py written.


In [6]:
# ── Fix: rewrite launcher.py with __main__ guard ────────────────────────────
launcher_code = (
    "import sys, os\n"
    "sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n"
    "\n"
    "if __name__ == '__main__':\n"
    "    import multiprocessing\n"
    "    multiprocessing.set_start_method('fork', force=True)\n"
    "\n"
    "    import app as app_module\n"
    "    app_module.init_cluster()\n"
    "\n"
    "    import uvicorn\n"
    "    uvicorn.run(app_module.app, host='0.0.0.0', port=8000, log_level='info')\n"
)

with open('launcher.py', 'w') as f:
    f.write(launcher_code)

print('✅ launcher.py rewritten.')
print(launcher_code)

✅ launcher.py rewritten.
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

if __name__ == '__main__':
    import multiprocessing
    multiprocessing.set_start_method('fork', force=True)

    import app as app_module
    app_module.init_cluster()

    import uvicorn
    uvicorn.run(app_module.app, host='0.0.0.0', port=8000, log_level='info')



In [9]:
# ── CELL 6: Set ngrok auth token ────────────────────────────────────────────
# Free token: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = '3CLdqKTCFWhzCcT4vy6KiVpii9x_3ucv42w1sMFDxeLZ2aUvf'
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print('✅ ngrok configured.')

✅ ngrok configured.


In [32]:
# ── CELL 7: LAUNCH ──────────────────────────────────────────────────────────
# launcher.py runs as a SUBPROCESS with no existing event loop.
# LocalCUDACluster and uvicorn both boot cleanly.
import subprocess, sys
from pyngrok import ngrok

ngrok.kill()
tunnel = ngrok.connect(8000)
print('\n' + '='*60)
print(f'  🌾 AgriCast is LIVE at: {tunnel.public_url}')
print('='*60 + '\n')
print('Logs stream below. Press ■ Stop to shut down.\n')

proc = subprocess.Popen(
    [sys.executable, 'launcher.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\n Server stopped.')


  🌾 AgriCast is LIVE at: https://ceramics-stays-overfill.ngrok-free.dev

Logs stream below. Press ■ Stop to shut down.

>> Initialising LocalCUDACluster ...
[ 14290][23:21:32:471780][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.
INFO:     Started server process [14290]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
   Dask ready -> http://127.0.0.1:8787/status
>> Loading 2025_fixed.csv ...
>> Data ready.
INFO:     2401:4900:1f2c:2572:935:8658:7aea:d26c:0 - "GET / HTTP/1.1" 200 OK
2026-06-26 23:21:36,644 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7f1e947ca46e03af99a76ca0c1b8e9cd initialized by task ('shuffle-transfer-7f1e947ca46e03af99a76ca0c1b8e9cd', 0) executed on worker tcp://127.0.0.1:37275
2026-06-26 23:21:36,694 - distributed.shuffle._scheduler_p

In [11]:
!find /content/drive -name "2025.csv"

find: ‘/content/drive’: No such file or directory


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
!find /content/drive -name "2025.csv"

In [16]:
from google.colab import files
uploaded = files.upload()  # upload the downloaded CSV

import os
# rename it to what the app expects
os.rename(list(uploaded.keys())[0], '/content/2025.csv')
print("✅ Done!")

Saving agmarknet_india_historical_prices_2024_2025.csv to agmarknet_india_historical_prices_2024_2025.csv
✅ Done!


In [18]:
import pandas as pd
df = pd.read_csv('/content/2025.csv', nrows=2)
print(df.columns.tolist())

['Sl no.', 'District Name', 'Market Name', 'Commodity', 'Variety', 'Grade', 'Min Price (Rs./Quintal)', 'Max Price (Rs./Quintal)', 'Modal Price (Rs./Quintal)', 'Price Date', 'State']


In [19]:
!grep -n "Arrival_Date\|District\|Modal_Price\|State" /content/app.py | head -30

39:    ddf_all["Arrival_Date"] = ddf_all["Arrival_Date"].map_partitions(
47:    ddf["month"]        = ddf["Arrival_Date"].dt.month
48:    ddf["day"]          = ddf["Arrival_Date"].dt.day
49:    ddf["day_of_year"]  = ddf["Arrival_Date"].dt.dayofyear
59:        ddf[f"lag_{lag}"] = ddf["Modal_Price"].shift(lag)
69:    states = sorted(ddf_all["State"].unique().compute().to_arrow().to_pylist())
75:    mask = ddf_all["State"] == state
76:    districts = sorted(ddf_all[mask]["District"].unique().compute().to_arrow().to_pylist())
82:    mask = (ddf_all["State"] == state) & (ddf_all["District"] == district)
93:    mask     = ((ddf_all["State"]     == state) &
94:                (ddf_all["District"]  == district) &
103:    y = ddf_proc["Modal_Price"].astype("float32")
108:    hist_month  = ddf_proc[month_mask][["day","Modal_Price","Min_Price","Max_Price"]].compute()
110:    actual_2025 = hist_month["Modal_Price"].to_arrow().to_pylist()
112:    all_prices = ddf_proc["Modal_Price"].compute()
128: 

In [20]:
!grep -n "read_csv\|FILE_PATH" /content/app.py | head -10

14:FILE_PATH   = "2025.csv"
33:    if not os.path.exists(FILE_PATH):
34:        raise RuntimeError(f"{FILE_PATH} not found.")
36:    print(f">> Loading {FILE_PATH} ...")
37:    ddf_all = dask_cudf.read_csv(FILE_PATH)


In [21]:
lines = open('/content/app.py').readlines()

rename_line = """    ddf_all = ddf_all.rename(columns={
        'Price Date': 'Arrival_Date',
        'District Name': 'District',
        'Modal Price (Rs./Quintal)': 'Modal_Price',
        'Min Price (Rs./Quintal)': 'Min_Price',
        'Max Price (Rs./Quintal)': 'Max_Price',
        'Market Name': 'Market'
    })\n"""

# Insert after line 37 (index 36)
lines.insert(37, rename_line)

with open('/content/app.py', 'w') as f:
    f.writelines(lines)

print("✅ Done!")

✅ Done!


In [23]:
!git add app.py
!git commit -m "Fix column names for new dataset"
!git push

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [24]:
import pandas as pd
df = pd.read_csv('/content/2025.csv', nrows=3)
print(df['Price Date'].values)

['05 Apr 2025' '14 Jun 2025' '23 Jun 2025']


In [25]:
content = open('/content/app.py').read()
content = content.replace("format=\"%Y-%m-%d\"", "format=\"%d %b %Y\"")
open('/content/app.py', 'w').write(content)
print("✅ Fixed!")

✅ Fixed!


In [27]:
content = open('/content/app.py').read()
content = content.replace(
    'format="%d %b %Y"',
    'format="%d-%m-%Y"'
)

# Also add a preprocessing step to convert the date format
old = '>> Loading {FILE_PATH} ...""")\n    ddf_all = dask_cudf.read_csv(FILE_PATH)'
open('/content/app.py', 'w').write(content)

# Better approach - convert dates using pandas first
fix = '''
import pandas as pd
df_temp = pd.read_csv('2025.csv')
df_temp['Price Date'] = pd.to_datetime(df_temp['Price Date'], format='%d %b %Y').dt.strftime('%d-%m-%Y')
df_temp.to_csv('2025_fixed.csv', index=False)
print("✅ Date format fixed!")
'''
exec(fix)

✅ Date format fixed!


In [28]:
content = open('/content/app.py').read()
content = content.replace('FILE_PATH   = "2025.csv"', 'FILE_PATH   = "2025_fixed.csv"')
open('/content/app.py', 'w').write(content)
print("✅ Done!")

✅ Done!


In [30]:
lines = open('/content/app.py').readlines()
for i, line in enumerate(lines, 1):
    if 'date' in line.lower() or 'format' in line.lower() or 'datetime' in line.lower():
        print(f"{i}: {line}", end='')

39:         'Price Date': 'Arrival_Date',
47:     ddf_all["Arrival_Date"] = ddf_all["Arrival_Date"].map_partitions(
48:         cudf.to_datetime, format="%d-%m-%Y",
49:         meta=cudf.Series([], dtype="datetime64[ns]")
55:     ddf["month"]        = ddf["Arrival_Date"].dt.month
56:     ddf["day"]          = ddf["Arrival_Date"].dt.day
57:     ddf["day_of_year"]  = ddf["Arrival_Date"].dt.dayofyear
135:     date_strs  = [f"2026-{month:02d}-{d:02d}" for d in days]
139:         "Arrival_Date": cudf.to_datetime(cudf.Series(date_strs), format="%d-%m-%Y"),


In [31]:
content = open('/content/app.py').read()
content = content.replace(
    '"Arrival_Date": cudf.to_datetime(cudf.Series(date_strs), format="%d-%m-%Y")',
    '"Arrival_Date": cudf.to_datetime(cudf.Series(date_strs), format="%Y-%m-%d")'
)
open('/content/app.py', 'w').write(content)
print("✅ Fixed!")

✅ Fixed!


In [34]:
!git add app.py
!git commit -m "Fix column and date format for new dataset"
!git push

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [35]:
!git init
!git remote add origin https://YOUR_GITHUB_TOKEN@github.com/sabyg147/AgriculturePrediction.git
!git add app.py launcher.py templates/ agri.ipynb .gitignore
!git commit -m "Add all project files"
!git branch -M main
!git push origin main --force

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
fatal: pathspec 'agri.ipynb' did not match any files
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@68828beda6da.(none)')
error: src refspec main does not match any
error: failed to push some refs t

In [ ]:
!git config --global user.email "sabyasachig147@gmail.com"
!git config --global user.name "sabyg147"